# 03 - Model Setup

Connect to Ollama and pick the model this machine can run well.

- Ollama itself decides GPU or CPU; the **same model file** runs on
  either. What we choose is the model **size**.
- At startup the app loads the small model and asks Ollama how much of
  it sits in GPU memory. No GPU -> CPU profile. GPU -> try the 7B and
  keep it only if it fits **entirely** (a partly offloaded 7B is often
  slower than a 3B that fits).
- `MODEL_PROFILE=auto|gpu|cpu` overrides the choice.

The notebook also checks schema-enforced JSON output and measures speed
for both models on GPU and on CPU (forced with `num_gpu=0`), so we know
what a CPU-only machine will see.

Becomes `src/helpers/model_selector.py` and the model part of
`config/config.py`.

In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import json
import logging
import os
import time
from pathlib import Path

import ollama
import pandas as pd


BASE_DIR = (
    Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
)
DATASETS_DIR = BASE_DIR / "data" / "output_data" / "datasets"
MODEL_OUTPUT_DIR = BASE_DIR / "data" / "output_data" / "model"

# Inside Docker this becomes http://host.docker.internal:11434
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
MODEL_PROFILE = os.getenv("MODEL_PROFILE", "auto")      # auto | gpu | cpu

# One entry per profile. The GPU profile can afford a bigger model,
# more retries and a longer context.
PROFILES = {
    "gpu": {
        "model": "qwen2.5:7b-instruct",
        "num_ctx": 8192,
        "max_retries": 3,
        "request_timeout": 120,
    },
    "cpu": {
        "model": "qwen2.5:3b-instruct",
        "num_ctx": 4096,
        "max_retries": 2,
        "request_timeout": 300,
    },
}

# A model "fits" when at least this share of it is in GPU memory.
GPU_FIT_SHARE = 0.99

LLM_TEMPERATURE = 0
KEEP_ALIVE = "30m"          # keep the chosen model loaded between questions
CONNECT_TIMEOUT = 10

# Speed test
BENCHMARK_RUNS = 2
BENCHMARK_MAX_TOKENS = 120

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("03_model_setup")

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ollama URL   :", OLLAMA_URL)
print("Profile mode :", MODEL_PROFILE)

Ollama URL   : http://localhost:11434
Profile mode : auto


## 2. Connecting to Ollama

Fail early with a clear message: server not running, or a model not
pulled yet (`ollama pull qwen2.5:3b-instruct`).

In [2]:
# ============================================================
# 2. CONNECTING TO OLLAMA
# ============================================================

def get_client(timeout=CONNECT_TIMEOUT):
    """Ollama client for OLLAMA_URL."""

    return ollama.Client(host=OLLAMA_URL, timeout=timeout)


def check_ollama(client):
    """
    Make sure the server answers and the profile models are installed.

    Returns:
        list of installed model names
    """

    # Step 1: the server must be reachable.
    try:
        installed = [model["model"] for model in client.list()["models"]]

    except Exception as exc:
        raise ConnectionError(
            f"Ollama is not reachable at {OLLAMA_URL}. Start it with "
            f"'ollama serve' (or the Ollama app). Details: {exc}"
        ) from exc

    # Step 2: every profile model must be pulled.
    missing = [settings["model"] for settings in PROFILES.values()
               if settings["model"] not in installed]

    if missing:
        raise ValueError(
            "Missing Ollama model(s): "
            + ", ".join(missing)
            + ". Pull with: "
            + " ; ".join(f"ollama pull {name}" for name in missing)
        )

    logger.info("Ollama reachable, %d model(s) installed.", len(installed))

    return installed


client = get_client()
installed_models = check_ollama(client)
installed_models

2026-11-14 14:58:43,774 | INFO | HTTP Request: GET http://localhost:11434/api/tags "HTTP/1.1 200 OK"


2026-11-14 14:58:43,775 | INFO | Ollama reachable, 8 model(s) installed.


['qwen2.5:3b-instruct',
 'qwen2.5:7b-instruct',
 'nomic-embed-text:latest',
 'bge-m3:latest',
 'qwen3-embedding:0.6b',
 'embeddinggemma:300m',
 'qwen3-vl:4b-instruct-q4_K_M',
 'mistral:latest']

## 3. Detecting GPU or CPU

`ollama ps` reports each loaded model's total `size` and `size_vram`
(the part in GPU memory). The model is loaded with the same `num_ctx`
it will be used with, because the context size changes how much memory
it needs.

In [3]:
# ============================================================
# 3. DETECTING GPU OR CPU
# ============================================================

def load_model(client, model, num_ctx, num_gpu=None):
    """Load a model into memory without generating anything."""

    options = {"num_ctx": num_ctx}

    if num_gpu is not None:
        options["num_gpu"] = num_gpu

    client.generate(model=model, prompt="", options=options,
                    keep_alive=KEEP_ALIVE)


def unload_model(client, model):
    """Free the memory a model uses."""

    client.generate(model=model, prompt="", keep_alive=0)


def gpu_share(client, model):
    """
    Share of a loaded model that sits in GPU memory (0.0 - 1.0).

    Returns None when the model is not loaded.
    """

    for loaded in client.ps()["models"]:
        if loaded["model"] == model and loaded["size"]:
            return loaded["size_vram"] / loaded["size"]

    return None


def detect_profile(client):
    """
    Pick "gpu" or "cpu" from what Ollama actually loaded.

    Returns:
        (profile name, details dict)
    """

    small = PROFILES["cpu"]
    large = PROFILES["gpu"]

    # Step 1: load the small model; no GPU memory used -> CPU machine.
    load_model(client, small["model"], small["num_ctx"])
    small_share = gpu_share(client, small["model"]) or 0.0

    details = {"small_model_gpu_share": round(small_share, 3)}

    if small_share == 0:
        details["reason"] = "No GPU memory used by the small model."
        return "cpu", details

    # Step 2: free the small model, then try the large one.
    unload_model(client, small["model"])
    load_model(client, large["model"], large["num_ctx"])
    large_share = gpu_share(client, large["model"]) or 0.0
    details["large_model_gpu_share"] = round(large_share, 3)

    if large_share >= GPU_FIT_SHARE:
        details["reason"] = f"{large['model']} fits entirely in GPU memory."
        return "gpu", details

    # Step 3: the large model only partly fits -> use the small one.
    unload_model(client, large["model"])
    details["reason"] = (f"{large['model']} only {large_share:.0%} in GPU "
                         "memory; the small model is faster.")

    return "cpu", details


def select_model(client, mode=MODEL_PROFILE):
    """
    Model settings for this machine.

    mode "gpu" / "cpu" skips detection; "auto" detects.
    """

    if mode not in {"auto", "gpu", "cpu"}:
        raise ValueError(f"MODEL_PROFILE must be auto, gpu or cpu, not {mode!r}")

    if mode == "auto":
        profile, details = detect_profile(client)
    else:
        profile, details = mode, {"reason": f"Forced by MODEL_PROFILE={mode}."}

    settings = {"profile": profile, **PROFILES[profile], "detection": details}

    logger.info("Model profile: %s -> %s (%s)",
                profile, settings["model"], details["reason"])

    return settings


started = time.perf_counter()
model_settings = select_model(client)
print(f"Detection took {time.perf_counter() - started:.1f} s")
model_settings

2026-11-14 14:58:48,022 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


2026-11-14 14:58:48,024 | INFO | HTTP Request: GET http://localhost:11434/api/ps "HTTP/1.1 200 OK"


2026-11-14 14:58:48,063 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


2026-11-14 14:58:53,058 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


2026-11-14 14:58:53,059 | INFO | HTTP Request: GET http://localhost:11434/api/ps "HTTP/1.1 200 OK"


2026-11-14 14:58:53,060 | INFO | Model profile: gpu -> qwen2.5:7b-instruct (qwen2.5:7b-instruct fits entirely in GPU memory.)


Detection took 9.3 s


{'profile': 'gpu',
 'model': 'qwen2.5:7b-instruct',
 'num_ctx': 8192,
 'max_retries': 3,
 'request_timeout': 120,
 'detection': {'small_model_gpu_share': 1.0,
  'large_model_gpu_share': 1.0,
  'reason': 'qwen2.5:7b-instruct fits entirely in GPU memory.'}}

## 4. Structured JSON output

`format=<JSON schema>` makes Ollama constrain the model's output to the
schema, which is far more reliable for a 7B / 3B model than asking for
JSON in the prompt. `chat_json()` is the one function every later
notebook uses to call the model.

In [4]:
# ============================================================
# 4. STRUCTURED JSON OUTPUT
# ============================================================

def timed_chat(settings, messages, schema, max_tokens=None, num_gpu=None):
    """
    One model call with the schema as the output format.

    max_tokens caps the output; a cap that is too low cuts the JSON off,
    so normal calls leave it unset.

    Returns:
        (raw text content, dict stats)
    """

    client = get_client(timeout=settings["request_timeout"])
    options = {"temperature": LLM_TEMPERATURE, "num_ctx": settings["num_ctx"]}

    if max_tokens is not None:
        options["num_predict"] = max_tokens

    if num_gpu is not None:
        options["num_gpu"] = num_gpu

    started = time.perf_counter()
    response = client.chat(
        model=settings["model"], messages=messages, format=schema,
        options=options, keep_alive=KEEP_ALIVE,
    )
    seconds = time.perf_counter() - started

    # Ollama reports durations in nanoseconds.
    eval_seconds = (response["eval_duration"] or 0) / 1e9

    stats = {
        "model": settings["model"],
        "seconds": round(seconds, 2),
        "load_seconds": round((response["load_duration"] or 0) / 1e9, 2),
        "prompt_tokens": response["prompt_eval_count"],
        "output_tokens": response["eval_count"],
        "tokens_per_second": round(response["eval_count"] / eval_seconds, 1)
        if eval_seconds else None,
    }

    return response["message"]["content"], stats


def chat_json(settings, messages, schema):
    """
    Call the model and return its answer parsed against `schema`.

    Raises ValueError when the output is not valid JSON or misses a
    required key; the retry loop in notebook 05 catches it.

    Returns:
        (dict answer, dict stats)
    """

    # Step 1: call the model.
    content, stats = timed_chat(settings, messages, schema)

    # Step 2: parse and check the required keys.
    try:
        answer = json.loads(content)

    except json.JSONDecodeError as exc:
        raise ValueError(f"Model did not return valid JSON: {content[:200]!r}") from exc

    missing = [key for key in schema.get("required", []) if key not in answer]

    if missing:
        raise ValueError(f"Model answer is missing keys {missing}: {answer}")

    return answer, stats


SQL_ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "sql": {"type": "string"},
        "explanation": {"type": "string"},
    },
    "required": ["sql", "explanation"],
}

schema_text = (DATASETS_DIR / "messy_sales" / "schema.txt").read_text(encoding="utf-8")

test_messages = [
    {"role": "system", "content":
        "You write one SQLite SELECT query that answers the question, "
        "using only this schema:\n\n" + schema_text},
    {"role": "user", "content": "How many orders were placed in Delhi?"},
]

answer, stats = chat_json(model_settings, test_messages, SQL_ANSWER_SCHEMA)
print(json.dumps(answer, indent=2))
print(stats)

2026-11-14 14:58:56,610 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


{
  "sql": "SELECT COUNT(*) FROM messy_sales WHERE city = 'Delhi';",
  "explanation": "This query counts the number of rows where the city is 'Delhi', giving us the total number of orders placed in Delhi."
}
{'model': 'qwen2.5:7b-instruct', 'seconds': 3.32, 'load_seconds': 0.21, 'prompt_tokens': 525, 'output_tokens': 54, 'tokens_per_second': 67.1}


## 5. Speed on GPU and on CPU

Both models, both devices, same prompt. `num_gpu=0` keeps every layer
on the CPU, which is what a machine without a GPU gets. Output is capped
at a fixed number of tokens so runs are comparable.

In [5]:
# ============================================================
# 5. SPEED ON GPU AND ON CPU
# ============================================================

def benchmark(profile, device, runs=BENCHMARK_RUNS):
    """Average speed of one model on one device ("gpu" or "cpu")."""

    settings = PROFILES[profile]
    num_gpu = 0 if device == "cpu" else None

    # Load first so the timed runs do not include loading.
    load_model(get_client(), settings["model"], settings["num_ctx"], num_gpu)

    # Output is capped, so it may be cut off: only the timing is used.
    results = [
        timed_chat(settings, test_messages, SQL_ANSWER_SCHEMA,
                   max_tokens=BENCHMARK_MAX_TOKENS, num_gpu=num_gpu)[1]
        for _ in range(runs)
    ]

    unload_model(get_client(), settings["model"])

    return {
        "model": settings["model"],
        "device": device,
        "seconds_per_answer": round(sum(r["seconds"] for r in results) / runs, 2),
        "tokens_per_second": round(
            sum(r["tokens_per_second"] or 0 for r in results) / runs, 1),
        "prompt_tokens": results[0]["prompt_tokens"],
    }


benchmarks = []

for profile in ("cpu", "gpu"):
    for device in ("gpu", "cpu"):
        benchmarks.append(benchmark(profile, device))
        logger.info("Benchmark: %s", benchmarks[-1])

benchmark_table = pd.DataFrame(benchmarks)
benchmark_table

2026-11-14 14:59:03,881 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


2026-11-14 14:59:07,545 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


2026-11-14 14:59:10,740 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


2026-11-14 14:59:13,008 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


2026-11-14 14:59:13,009 | INFO | Benchmark: {'model': 'qwen2.5:3b-instruct', 'device': 'gpu', 'seconds_per_answer': 3.2, 'tokens_per_second': 105.2, 'prompt_tokens': 525}


2026-11-14 14:59:17,604 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


... (11 outputs left out)

2026-11-14 15:00:57,720 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


2026-11-14 15:01:00,119 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


2026-11-14 15:01:00,120 | INFO | Benchmark: {'model': 'qwen2.5:7b-instruct', 'device': 'cpu', 'seconds_per_answer': 26.52, 'tokens_per_second': 6.6, 'prompt_tokens': 525}


,model,device,seconds_per_answer,tokens_per_second,prompt_tokens
0,qwen2.5:3b-instruct,gpu,3.20,105.2,525
1,qwen2.5:3b-instruct,cpu,9.16,14.9,525
2,qwen2.5:7b-instruct,gpu,3.35,62.1,525
3,qwen2.5:7b-instruct,cpu,26.52,6.6,525


## 6. Checks

In [6]:
# ============================================================
# 6. CHECKS
# ============================================================

# Detection result is a known profile with complete settings.
assert model_settings["profile"] in PROFILES
assert {"model", "num_ctx", "max_retries", "request_timeout"} <= set(model_settings)

# Forcing a profile skips detection.
assert select_model(client, "cpu")["model"] == PROFILES["cpu"]["model"]
assert select_model(client, "gpu")["model"] == PROFILES["gpu"]["model"]

try:
    select_model(client, "fast")
    raise AssertionError("invalid MODEL_PROFILE was accepted")
except ValueError:
    pass

# Structured output: valid JSON with a SELECT query.
assert answer["sql"].strip().upper().startswith(("SELECT", "WITH"))

# A missing model gives a clear error, not a crash later.
PROFILES["cpu"]["model"] = "not-a-real-model:1b"
try:
    check_ollama(client)
    raise AssertionError("missing model was not reported")
except ValueError as exc:
    assert "ollama pull not-a-real-model:1b" in str(exc)
finally:
    PROFILES["cpu"]["model"] = "qwen2.5:3b-instruct"

# The GPU should beat the CPU for the same model.
speed = benchmark_table.set_index(["model", "device"])["tokens_per_second"]
for model in benchmark_table["model"].unique():
    assert speed[(model, "gpu")] > speed[(model, "cpu")], model

print("All checks passed.")

2026-11-14 15:01:00,272 | INFO | Model profile: cpu -> qwen2.5:3b-instruct (Forced by MODEL_PROFILE=cpu.)


2026-11-14 15:01:00,273 | INFO | Model profile: gpu -> qwen2.5:7b-instruct (Forced by MODEL_PROFILE=gpu.)


2026-11-14 15:01:02,302 | INFO | HTTP Request: GET http://localhost:11434/api/tags "HTTP/1.1 200 OK"


All checks passed.


## 7. Save and reload the chosen model

In [7]:
# ============================================================
# 7. SAVE AND RELOAD THE CHOSEN MODEL
# ============================================================

# The benchmark unloaded everything; load the chosen model again.
load_model(client, model_settings["model"], model_settings["num_ctx"])

output = {
    "ollama_url": OLLAMA_URL,
    "selected": model_settings,
    "benchmarks": benchmarks,
}

with (MODEL_OUTPUT_DIR / "model_profile.json").open("w", encoding="utf-8") as file:
    json.dump(output, file, indent=2)

print(json.dumps(output["selected"], indent=2))

2026-11-14 15:01:07,228 | INFO | HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


{
  "profile": "gpu",
  "model": "qwen2.5:7b-instruct",
  "num_ctx": 8192,
  "max_retries": 3,
  "request_timeout": 120,
  "detection": {
    "small_model_gpu_share": 1.0,
    "large_model_gpu_share": 1.0,
    "reason": "qwen2.5:7b-instruct fits entirely in GPU memory."
  }
}
